# DeepLOB "Super Model" (DeepLOB Encoder + Seq2Seq Decoder)

This notebook implements the architecture inspired by the MHF paper.

**Architecture:**
1.  **Encoder:** DeepLOB (Convolutional Blocks + Inception Module + LSTM) to extract rich spatial features from the LOB.
2.  **Decoder:** Seq2Seq LSTM with Additive Attention to generate the 60-second midprice path step-by-step.


In [47]:
# Maaz
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# Joe
import polars as pl 
import numpy as np
import torch.nn as nn
from torch import optim
import torch
import re
from tqdm import trange
import random
from utils import *
from data.simulate_walk_the_book import *
import warnings


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Fix randomness for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


device: cuda


In [48]:
# Paths and volume_to_fill
root = Path("data") if Path("data").exists() else Path.cwd()
import sys
if str(root / "src") not in sys.path:
    sys.path.append(str(root / "src"))

symbol_dir = root / "BTCUSDT"
X_path = symbol_dir / "X_train.parquet"
Y_path = symbol_dir / "y_train.parquet"
vol_path = symbol_dir / "vol_to_fill.txt"

volume_to_fill = None
if vol_path.exists():
    import re
    with open(vol_path) as f:
        m = re.search(r"([\d.]+)", f.read())
    if m:
        volume_to_fill = float(m.group(1))
print("volume_to_fill:", volume_to_fill)


volume_to_fill: 4.0


# Preprocessing

In [49]:
# Run this cell to see specifics on preprocessing
help(preprocess)

Help on function preprocess in module utils:

preprocess(X, y, device) -> tuple
    Clean, validate, normalize, and tensorize feature and target datasets.
    
    This function applies a deterministic preprocessing pipeline to input
    feature (`X`) and target (`y`) Polars DataFrames. The pipeline enforces
    strict temporal consistency per instrument, handles missing values using
    market-aware rules, removes invalid instruments, converts data into
    fixed-length PyTorch tensors, and applies z-score normalization.
    
    Processing steps:
    1. Sort data by instrument ID and time.
    2. Backfill only *leading* NaNs in price columns per instrument.
    3. Forward-fill remaining price NaNs.
    4. Replace missing volume values with zeros.
    5. Remove instruments with:
       - duplicate (ID, time) pairs
       - missing timesteps (incomplete sequences)
    6. Convert DataFrames into tensors of shape:
       - X: (input_seq_len, num_ids, num_features)
       - y: (target_seq

## 1. The Encoder: DeepLOB (Heavy)
This is the standard DeepLOB CNN-Inception encoder.

In [50]:
LOB_COLS = []
for i in range(1, 6):
    LOB_COLS.append(f"ask_price_{i}")
    LOB_COLS.append(f"ask_vol_{i}")
    LOB_COLS.append(f"bid_price_{i}")
    LOB_COLS.append(f"bid_vol_{i}")

FEATURE_COLS = LOB_COLS

# Verification print
print(f"CNN Input Width: 4 (Columns: Price/Vol/Price/Vol)")
print(f"CNN Input Height: 5 (Rows: Levels 1 through 5)")
print(f"Total Features: {len(FEATURE_COLS)}")

CNN Input Width: 4 (Columns: Price/Vol/Price/Vol)
CNN Input Height: 5 (Rows: Levels 1 through 5)
Total Features: 20


In [51]:
class DeepLOBEncoder(nn.Module):
    def __init__(self, in_ch: int = 1):
        super().__init__()
        # Conv Blocks 1, 2, 3 (Keep these exactly as you have them)
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_ch, 32, kernel_size=(1,2), stride=(1,1)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32,32,kernel_size=(3,1), padding=(1,0)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32,32,kernel_size=(3,1), padding=(1,0)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32,32,kernel_size=(1,2),stride=(1,1)), nn.Tanh(), nn.BatchNorm2d(32),
            nn.Conv2d(32,32,kernel_size=(3,1), padding=(1,0)), nn.Tanh(), nn.BatchNorm2d(32),
            nn.Conv2d(32,32,kernel_size=(3,1), padding=(1,0)), nn.Tanh(), nn.BatchNorm2d(32),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32,32,kernel_size=(1,2)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32,32,kernel_size=(3,1), padding=(1,0)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32,32,kernel_size=(3,1), padding=(1,0)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
        )
        # Inception Modules (Keep these exactly as you have them)
        self.inp1 = nn.Sequential(
            nn.Conv2d(32,64,kernel_size=(1,1),padding='same'), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
            nn.Conv2d(64,64,kernel_size=(3,1),padding='same'), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
        )
        self.inp2 = nn.Sequential(
            nn.Conv2d(32,64,kernel_size=(1,1),padding='same'), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
            nn.Conv2d(64,64,kernel_size=(5,1),padding='same'), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
        )
        self.inp3 = nn.Sequential(
            nn.MaxPool2d((3,1),stride=(1,1),padding=(1,0)),
            nn.Conv2d(32,64,kernel_size=(1,1),padding='same'), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
        )

    def forward(self, x):
        # x: [Batch, Time, Features]
        B, T, F = x.shape
        
        # 1. Standardize to 20 features (5 levels * 4)
        x = x[:, :, :20] 
        
        # 2. Reshape for CNN: [B*T, 1, 5, 4]
        x = x.reshape(B * T, 1, 5, 4) 
        
        # 3. Convolutional Stages
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        
        # 4. Inception Stage
        x = torch.cat([self.inp1(x), self.inp2(x), self.inp3(x)], dim=1) # [B*T, 192, 5, 1]
        
        # 5. Global Average Pooling (The Fix)
        # This collapses the '5' (height) into 1, giving us [B*T, 192, 1, 1]
        x = torch.mean(x, dim=(2, 3)) 
        
        # 6. Final Shape for LSTM: [Batch, Time, 192]
        return x.view(B, T, 192)

## 2. The Decoder: Seq2Seq Attention
This replaces the simple Bi-LSTM head with a full autoregressive decoder.

In [52]:
class AdditiveAttention(nn.Module):
    def __init__(self, enc_dim: int, dec_dim: int, attn_dim: int = 64):
        super().__init__()
        self.W_e = nn.Linear(enc_dim, attn_dim, bias=False)
        self.W_d = nn.Linear(dec_dim, attn_dim, bias=False)
        self.v = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, enc_outputs: torch.Tensor, dec_state: torch.Tensor) -> torch.Tensor:
        # enc_outputs: [B, T, E], dec_state: [B, D]
        e = self.W_e(enc_outputs) + self.W_d(dec_state).unsqueeze(1)  # [B, T, A]
        scores = self.v(torch.tanh(e)).squeeze(-1)  # [B, T]
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(weights.unsqueeze(-1) * enc_outputs, dim=1)  # [B, E]
        return context

class SuperModel(nn.Module):
    def __init__(self, hidden: int = 128, horizon: int = 60, dropout: float=0.1, ask_bid_idx=(0, 2)):
        super().__init__()
        # 1. Encoder Part
        self.spatial = DeepLOBEncoder(in_ch=1) # The Heavy CNN [B, T, 192]
        self.encoder = nn.LSTM(
            input_size=192, # DeepLOB output dim
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        
        # 2. Decoder Part
        self.attn = AdditiveAttention(enc_dim=2 * hidden, dec_dim=hidden)
        self.decoder = nn.LSTM(
            input_size=1 + 2 * hidden,  # previous y + context
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            dropout=0.0,
        )
        self.out = nn.Linear(hidden, 1)
        self.init_h = nn.Linear(2 * hidden, hidden)
        self.init_c = nn.Linear(2 * hidden, hidden)
        self.horizon = horizon

        self.ask_idx, self.bid_idx = ask_bid_idx

    def forward(self, x: torch.Tensor, y_teacher: torch.Tensor = None) -> torch.Tensor:
        # x: [B, T, F]
        # y_teacher: [B, 60]
        
        # Encode
        h_spatial = self.spatial(x)  # [B, T, 192]
        enc_out, _ = self.encoder(h_spatial)  # [B, T, 2H]

        # Init decoder state from encoder mean
        enc_mean = enc_out.mean(dim=1)
        dec_h0 = torch.tanh(self.init_h(enc_mean))
        dec_c0 = torch.tanh(self.init_c(enc_mean))
        dec_h = dec_h0.unsqueeze(0)  # [1, B, H]
        dec_c = dec_c0.unsqueeze(0)

        outputs = []
        # Seed: use last encoder input midprice proxy (col 0 normalized)
        last_ask = x[:, -1, self.ask_idx]
        last_bid = x[:, -1, self.bid_idx]
        prev_y = ((last_ask + last_bid) / 2.0).unsqueeze(-1) # [B, 1]

        for t in range(self.horizon):
            context = self.attn(enc_out, dec_h[-1])  # [B, 2H]
            dec_in = torch.cat([prev_y, context], dim=1).unsqueeze(1)  # [B, 1, 1+2H]
            dec_out, (dec_h, dec_c) = self.decoder(dec_in, (dec_h, dec_c))
            y_hat = self.out(dec_out).squeeze(1)  # [B, 1]
            outputs.append(y_hat.squeeze(-1))
            if self.training and y_teacher is not None:
                # Teacher forcing
                prev_y = y_teacher[:, t].unsqueeze(-1)
            else:
                # Autoregressive
                prev_y = y_hat.detach()

        return torch.stack(outputs, dim=1)  # [B, 60]

def forecast_loss(pred, target, smooth_lambda=0.02):
    mse = (pred - target) ** 2
    loss = mse.mean()
    if smooth_lambda > 0:
        smooth = (pred[:, 1:] - pred[:, :-1]) ** 2
        loss = loss + smooth_lambda * smooth.mean()
    return loss


## 3. Dataset & Training Loop

In [53]:
from dataclasses import dataclass
import pandas as pd
import polars as pl
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

@dataclass
class TrainCfg:
    epochs: int = 15
    batch_size: int = 16
    lr: float = 1e-3
    weight_decay: float = 1e-5
    smooth_lambda: float = 0.02
    val_ratio: float = 0.2
    input_window: int = 300   # Look-back period
    target_window: int = 60   # Prediction horizon

class TensorTimeDataset(Dataset):
    def __init__(self, X, Y_full, Y_mid, input_window: int):
        self.X = X      
        self.Y = Y_full 
        self.Y_mid = Y_mid 
        self.input_window = input_window

    def __len__(self):
        return self.X.shape[1]

    def __getitem__(self, idx):
        # Dynamically slice based on config input_window
        x_sample = self.X[-self.input_window:, idx, :] 
        y_full_sample = self.Y[:, idx, :]
        target = self.Y_mid[idx, :]
        return x_sample, y_full_sample, target

def chrono_split(x_df, y_df, val_ratio=0.2):
    ids = np.sort(x_df["anonymized_id"].unique())
    split = int(len(ids) * (1 - val_ratio))
    tr_ids, va_ids = ids[:split], ids[split:]
    
    x_tr = x_df[x_df["anonymized_id"].isin(tr_ids)].reset_index(drop=True)
    x_va = x_df[x_df["anonymized_id"].isin(va_ids)].reset_index(drop=True)
    y_tr = y_df[y_df["anonymized_id"].isin(tr_ids)].reset_index(drop=True)
    y_va = y_df[y_df["anonymized_id"].isin(va_ids)].reset_index(drop=True)
    return x_tr, x_va, y_tr, y_va

def train_epoch(model, loader, opt, cfg: TrainCfg):
    model.train()
    total_loss, n = 0.0, 0
    for xb, _, target in loader:
        xb, target = xb.to(device), target.to(device)
        opt.zero_grad()
        
        # SuperModel handles teacher forcing internally if y_teacher is provided
        pred = model(xb, y_teacher=target)
        loss = forecast_loss(pred, target, cfg.smooth_lambda)
        
        loss.backward()
        opt.step()
        
        total_loss += float(loss.item()) * xb.size(0)
        n += xb.size(0)
    return total_loss / max(n, 1)

def eval_epoch(model, loader, cfg: TrainCfg):
    model.eval()
    total_loss, n = 0.0, 0
    with torch.no_grad():
        for xb, _, target in loader:
            xb, target = xb.to(device), target.to(device)
            # No teacher forcing during evaluation
            pred = model(xb, y_teacher=None)
            loss = forecast_loss(pred, target, cfg.smooth_lambda)
            total_loss += float(loss.item()) * xb.size(0)
            n += xb.size(0)
    return total_loss / max(n, 1)

def train_val(cfg: TrainCfg = TrainCfg()):
    # 1. Load Data
    X_raw = pd.read_parquet(X_path).sort_values(["anonymized_id", "time_in_hour"])
    Y_raw = pd.read_parquet(Y_path).sort_values(["anonymized_id", "time_in_hour"])
    
    # 2. Split
    x_tr_pd, x_va_pd, y_tr_pd, y_va_pd = chrono_split(X_raw, Y_raw, val_ratio=cfg.val_ratio)

    # Insert before section 3 in train_val
    raw_mid_price = (y_tr_pd["ask_price_1"] + y_tr_pd["bid_price_1"]) / 2.0
    actual_mid_mean = raw_mid_price.mean()
    actual_mid_std = raw_mid_price.std()

    # 3. Preprocess (to Polars -> to Tensors)
    X_tr_tens, Y_tr_tens, tr_means, tr_stds, _, _, f_map = preprocess(
        pl.from_pandas(x_tr_pd), pl.from_pandas(y_tr_pd), device
    )
    X_va_tens, Y_va_tens, _, _, _, y_va_map, _ = preprocess(
        pl.from_pandas(x_va_pd), pl.from_pandas(y_va_pd), device
    )

    # --- THE FIX: ENFORCE FEATURE_COLS ORDER ---
    # This ensures index 0-19 match the [P, V, P, V] pattern DeepLOB expects
    feat_indices = [f_map[col] for col in FEATURE_COLS]
    
    X_tr_tens = X_tr_tens[:, :, feat_indices]
    X_va_tens = X_va_tens[:, :, feat_indices]
    
    # Also re-order the means/stds so they match the new tensor indices
    tr_means = tr_means[:, :, feat_indices]
    tr_stds  = tr_stds[:, :, feat_indices]
    # -------------------------------------------
    
    # 4. Target Mid-Price extraction
    # Note: Because we re-ordered above, ask_price_1 is now index 0, bid_price_1 is index 2
    # But using the re-mapped indices via f_map logic is still safer:
    new_f_map = {col: i for i, col in enumerate(FEATURE_COLS)}
    a_idx, b_idx = new_f_map["ask_price_1"], new_f_map["bid_price_1"]
    
    mid_tr = (Y_tr_tens[:, :, f_map["ask_price_1"]] + Y_tr_tens[:, :, f_map["bid_price_1"]]) / 2.0
    mid_va = (Y_va_tens[:, :, f_map["ask_price_1"]] + Y_va_tens[:, :, f_map["bid_price_1"]]) / 2.0

    mid_mean = float(mid_tr.mean())
    mid_std  = float(mid_tr.std()) if mid_tr.std() != 0 else 1e-6

    # 5. Datasets (passing cfg.input_window)
    train_ds = TensorTimeDataset(X_tr_tens, Y_tr_tens, mid_tr.T, input_window=cfg.input_window)
    val_ds   = TensorTimeDataset(X_va_tens, Y_va_tens, mid_va.T, input_window=cfg.input_window)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False)

    # 6. Model & Training
    model = SuperModel(
        horizon=cfg.target_window, 
        ask_bid_idx=(a_idx, b_idx) # Use the indices relative to FEATURE_COLS
    ).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    for epoch in range(cfg.epochs):
        tr_l = train_epoch(model, train_loader, optimizer, cfg)
        va_l = eval_epoch(model, val_loader, cfg)
        print(f"Epoch {epoch+1:02d} | Train: {tr_l:.6f} | Val: {va_l:.6f}")

    scalers = {
        "feat_means": tr_means, 
        "feat_stds": tr_stds, 
        "mid_mean": actual_mid_mean, # Use raw dollar values
        "mid_std": actual_mid_std     # Use raw dollar values
    }
    return model, scalers, val_loader, y_va_map


In [54]:
# --- Execution ---
config = TrainCfg(
    epochs=20, 
    batch_size=32, 
    input_window=60*60, # -> input window of 5 minutes
    smooth_lambda=0.01
)

model, scalers, val_loader, va_id_map = train_val(config)

Epoch 01 | Train: 1.143226 | Val: 3.054393
Epoch 02 | Train: 0.186689 | Val: 4.259633
Epoch 03 | Train: 0.067022 | Val: 0.749671
Epoch 04 | Train: 0.046122 | Val: 0.850783
Epoch 05 | Train: 0.041983 | Val: 0.680866
Epoch 06 | Train: 0.035986 | Val: 0.658451
Epoch 07 | Train: 0.028969 | Val: 0.489817
Epoch 08 | Train: 0.028788 | Val: 0.772962
Epoch 09 | Train: 0.030651 | Val: 0.318883
Epoch 10 | Train: 0.025524 | Val: 0.374477
Epoch 11 | Train: 0.021428 | Val: 0.421324
Epoch 12 | Train: 0.018559 | Val: 0.235891
Epoch 13 | Train: 0.016643 | Val: 0.420873
Epoch 14 | Train: 0.015573 | Val: 0.419082
Epoch 15 | Train: 0.014821 | Val: 0.366581
Epoch 16 | Train: 0.013597 | Val: 0.159199
Epoch 17 | Train: 0.013297 | Val: 0.176920
Epoch 18 | Train: 0.012164 | Val: 0.253954
Epoch 19 | Train: 0.011195 | Val: 0.257111
Epoch 20 | Train: 0.010658 | Val: 0.360171


## 4. Evaluation
Calculate implementation error (bps) usage `alpha=1.0` (pure model).

In [57]:
# Combined Evaluation & R^2 Calculation (Robust Version)
run_full_eval = True

if run_full_eval and 'model' in globals():
    from sklearn.metrics import r2_score
    from tqdm import tqdm
    import numpy as np
    
    # 1. Setup Feature Indices (Matches DeepLOB interleaving: P, V, P, V)
    LOB_COLS = []
    for i in range(1, 6):
        LOB_COLS.extend([f"ask_price_{i}", f"ask_vol_{i}", f"bid_price_{i}", f"bid_vol_{i}"])

    model.eval()
    
    # Storage
    all_preds_z = []
    all_targets_z = []
    model_errs_bps = []
    
    # Indices for Mid-Price calculation (Ask 1 and Bid 1)
    # These are indices relative to the LOB_COLS list
    a1_idx = 0  # ask_price_1
    b1_idx = 2  # bid_price_1

    # We use the raw DataFrames to iterate by ID
    va_ids_array = va_id_map.cpu().numpy().flatten()
    id_to_idx = {int(uid): i for i, uid in enumerate(va_ids_array)}
    valid_eval_ids = [hid for hid in va_ids if int(hid) in id_to_idx]

    print(f"Running robust combined eval on {len(valid_eval_ids)} instruments...")

    for hid in tqdm(valid_eval_ids):
        idx = id_to_idx[int(hid)]
        
        # Get hour-specific data
        xh_raw = x_va_raw[x_va_raw["anonymized_id"] == hid]
        yh_raw = y_va_raw[y_va_raw["anonymized_id"] == hid]
        
        if len(yh_raw) < config.target_window:
            continue
            
        # 2. Get Instrument-Specific Scalers
        h_feat_means = scalers['feat_means'][0, idx, :].cpu().numpy()
        h_feat_stds  = scalers['feat_stds'][0, idx, :].cpu().numpy()
        
        # Determine Mid-Price scalers for this specific instrument
        # We average the Ask1 and Bid1 stats to get a proxy for Mid-Price stats
        h_mid_mean = (h_feat_means[a1_idx] + h_feat_means[b1_idx]) / 2.0
        h_mid_std  = (h_feat_stds[a1_idx] + h_feat_stds[b1_idx]) / 2.0
        
        # 3. Prepare Input
        xh = xh_raw.tail(config.input_window)
        x_arr = xh[LOB_COLS].astype(np.float32).to_numpy()
        
        # Z-Score the input features
        x_arr = (x_arr - h_feat_means) / (h_feat_stds + 1e-9)
        xb = torch.from_numpy(x_arr).unsqueeze(0).to(device)
        
        # 4. Predict
        with torch.no_grad():
            pred_z_batch = model(xb, y_teacher=None) 
            pred_z = pred_z_batch.cpu().numpy().flatten() # [60]

        # 5. Target Extraction (Z-Space)
        yh_future = yh_raw.head(config.target_window)
        target_mid_raw = (yh_future["ask_price_1"] + yh_future["bid_price_1"]) / 2.0
        
        # Normalize target using the SAME instrument-specific stats
        target_z = (target_mid_raw.to_numpy() - h_mid_mean) / (h_mid_std + 1e-9)

        # Store for Stats
        all_preds_z.append(pred_z)
        all_targets_z.append(target_z)

        # 6. Trading Simulation (Dollar Space)
        # Convert prediction to dollars using this instrument's scale
        mid_pred_dollars = (pred_z * h_mid_std) + h_mid_mean
        positions = schedule_from_forecasts(mid_pred_dollars, volume_to_fill)
        
        if positions.sum() > 0:
            positions = positions * (volume_to_fill / positions.sum())
        
        ask_p = yh_future[[f"ask_price_{l}" for l in range(1,6)]].to_numpy()
        ask_v = yh_future[[f"ask_vol_{l}" for l in range(1,6)]].to_numpy()
        bid_p = yh_future[[f"bid_price_{l}" for l in range(1,6)]].to_numpy()
        bid_v = yh_future[[f"bid_vol_{l}" for l in range(1,6)]].to_numpy() 
        
        total_vol, avg_p = simulate_walk_the_book(positions, ask_p, ask_v, bid_p, bid_v)
        
        # Trading BPS calculation
        bench_p = (ask_p[-1, 0] + bid_p[-1, 0]) / 2.0
        if total_vol > 0 and not np.isnan(avg_p):
            rel_err = abs(avg_p - bench_p) / bench_p
            model_errs_bps.append(rel_err * 10000.0)

    # --- FINAL REPORTING ---
    print("\n" + "="*40)
    print(f"RESULTS FOR {len(model_errs_bps)} EVALUATED HOURS")
    print("-" * 40)

    # Convert lists to matrices [N, 60]
    preds_z_mat = np.vstack(all_preds_z)
    targets_z_mat = np.vstack(all_targets_z)
    
    # 1. Z-Space R2 (Directional accuracy at level)
    f_preds = preds_z_mat.flatten()
    f_targets = targets_z_mat.flatten()
    mask = np.isfinite(f_preds) & np.isfinite(f_targets)
    
    r2_z = r2_score(f_targets[mask], f_preds[mask])
    corr_z = np.corrcoef(f_targets[mask], f_preds[mask])[0, 1]
    
    print(f"Z-Space R^2:      {r2_z:.6f}")
    print(f"Z-Space Corr:     {corr_z:.4f}")

    # 2. Return-Space R2 (Capturing the move)
    # We calculate diff on Z-scores directly to avoid dollar-scaling issues
    # This measures: "Did the price move X standard deviations as predicted?"
    pred_ret = np.diff(preds_z_mat, axis=1).flatten()
    targ_ret = np.diff(targets_z_mat, axis=1).flatten()
    
    ret_mask = np.isfinite(pred_ret) & np.isfinite(targ_ret)
    r2_ret = r2_score(targ_ret[ret_mask], pred_ret[ret_mask])
    corr_ret = np.corrcoef(targ_ret[ret_mask], pred_ret[ret_mask])[0, 1]
    
    print(f"Returns R^2:      {r2_ret:.6f}")
    print(f"Returns Corr:     {corr_ret:.4f}")
    
    # 3. Trading Performance
    if model_errs_bps:
        print(f"Mean Trading BPS: {np.mean(model_errs_bps):.4f}")
    print("="*40)

Running robust combined eval on 136 instruments...


100%|██████████| 136/136 [00:03<00:00, 35.37it/s]


RESULTS FOR 21 EVALUATED HOURS
----------------------------------------
Z-Space R^2:      0.002264
Z-Space Corr:     0.4090
Returns R^2:      -2.611231
Returns Corr:     -0.0242
Mean Trading BPS: 1.0125
